# TRACK-FA Entropy and Mutual-Information Feature Selection

This legacy analysis notebook audits entropy and mutual-information signals in TRACK-FA imaging features.

**Scope**

- Mutual information is used to study visit/progression separation.
- Clinical scores are kept out of model training.
- The newer `feature_selection_pipeline.ipynb` is the preferred notebook for fold-safe selection comparisons.

**How to read this notebook:** treat these analyses as feature-ranking diagnostics, not as final model-selection evidence unless the ranking is recomputed inside each training fold.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve project imports and data paths from either the repo root or notebooks folder.
def find_project_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not find the project root containing src/ and data/.")


_REPO_ROOT = find_project_root(Path.cwd())
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from src.config import set_global_seeds  # noqa: E402
from src.data.qc import standardize_train_test  # noqa: E402
from src.eval.cv import lda_loocv  # noqa: E402
from src.eval.metrics import (  # noqa: E402
    bootstrap_ci_d,
    compute_cohens_d,
    compute_srm,
    paired_deltas_from_long,
)
from src.features.entropy import (  # noqa: E402
    mi_feature_vs_binary_label,
    rank_features_by_mi,
)
from src.features.selection import (  # noqa: E402
    _global_rank,
    make_selection_fn,
    select_topk_global,
    select_topk_by_group,
)


## 1. Load Data and Shared Settings

This cell prepares the processed TRACK-FA table, feature lists, and reproducibility settings used by the entropy audit.


In [ ]:
set_global_seeds(42)

from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long

RANDOM_SEED = 42
N_BOOT = 2000  # Number of bootstrap resamples for confidence intervals.
K_VALUES = [1, 2, 3, 5]
CV_N_SPLITS = 5

DATA_PATH = _REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
# Load TRACK-FA paired rows, infer imaging groups, and convert to visit-level rows.
pairs = pd.read_csv(DATA_PATH)
groups = infer_trackfa_feature_groups(pairs)
df_long = trackfa_pairs_to_long(pairs)
subject_col = "pair_id"

structural = [c for c in list(groups.poms + groups.brainspinemorph) if c in df_long.columns]
diffusion = [c for c in list(groups.braindti) if c in df_long.columns]
ALL_FEATURES = sorted(set(structural + diffusion))

# Reporting groups for mutual-information summaries only; demographics stay out of imaging models.
FEATURE_GROUPS = {
    "structural": structural,
    "diffusion": diffusion,
}

paired_subjects = df_long.groupby(subject_col)["visit"].nunique()
paired_subjects = paired_subjects[paired_subjects == 2].index

print("Pairs shape:", pairs.shape)
print("Long shape:", df_long.shape)
print("Pair column:", subject_col, "| n_pairs:", df_long[subject_col].nunique())
print("Imaging feature counts:", {"all_imaging": len(ALL_FEATURES), **{k: len(v) for k, v in FEATURE_GROUPS.items()}})
print("Clinical scores retained only for benchmarks:", [c for c in ["FARS", "SARA", "ADL"] if c in df_long.columns])
print("Paired rows for progression:", len(paired_subjects))


## 2. Score Single-Feature Progression

Each imaging feature is evaluated as a simple progression signal. This gives a transparent baseline before any multivariable composite is fitted.


In [ ]:
feature_memberships = {}
for g, feats in FEATURE_GROUPS.items():
    for f in feats:
        feature_memberships.setdefault(f, set()).add(g)

features_for_mi = ALL_FEATURES

mi_visit_rows = []
visit_y = (df_long["visit"].values == 2).astype(int)
for f in features_for_mi:
    if f not in df_long.columns:
        mi, n = (np.nan, 0)
    else:
        mi, n = mi_feature_vs_binary_label(df_long[f], visit_y, is_discrete=(f == "sex"))
    mi_visit_rows.append({"feature": f, "mi_visit": mi, "n_visit": n})
mi_df = pd.DataFrame(mi_visit_rows)

rows_e = []
for _, r in mi_df.iterrows():
    f = r["feature"]
    groups = sorted(feature_memberships.get(f, []))
    if not groups:
        continue
    for g in groups:
        rows_e.append({
            "feature": f, "group": g,
            "mi_visit": r.get("mi_visit"), "n_visit": r.get("n_visit"),
        })
single_feature_entropy_df = pd.DataFrame(rows_e)
display(single_feature_entropy_df)


## 3. Estimate Mutual Information

Mutual information measures how much an imaging feature helps distinguish visit labels. It is useful for ranking, but global ranking can be optimistic if used before cross-validation.


In [ ]:
single_rows = []
paired = df_long[df_long[subject_col].isin(paired_subjects)].copy()
for f in ALL_FEATURES:
    if f not in paired.columns:
        continue
    tmp = paired[[subject_col, "visit", f]].dropna().copy()
    if tmp.empty:
        continue
    deltas = paired_deltas_from_long(tmp.rename(columns={f: "value"}), subject_col, "visit", "value")
    d_out = compute_cohens_d(deltas)
    srm_out = compute_srm(deltas)
    tmp_oof = tmp.rename(columns={f: "value"})
    _, d_lo, d_hi = bootstrap_ci_d(tmp_oof, subject_col, "visit", "value", n_boot=N_BOOT, seed=RANDOM_SEED)
    if f in background:
        g = "background"
    elif f in structural:
        g = "structural"
    elif f in structural_ext:
        g = "structural_ext"
    elif f in diffusion:
        g = "diffusion"
    else:
        g = "unknown"
    single_rows.append({
        "feature": f, "group": g,
        "d_feature": d_out["d"], "srm_feature": srm_out["srm"],
        "d_ci_low": d_lo, "d_ci_high": d_hi,
        "n_subjects": d_out["n"],
        "mean_diff": d_out["mean"], "sd_diff": d_out["sd"],
    })
single_feature_d_df = pd.DataFrame(single_rows).sort_values("d_feature", ascending=False)
display(single_feature_d_df)


## 4. Audit Rankings on the All-Imaging Pool

This section checks which features rank highly across the full imaging set. Use it for interpretation and sanity checks, not as a leakage-safe final selector.


In [ ]:
audit_rows = []
for source in ["mi_visit", "mi_fars1", "mi_dfars"]:
    ranked = _global_rank(single_feature_entropy_df, source)
    audit_rows.append({
        "entropy_source": source,
        "n_ranked_features": len(ranked),
        "top_features": ranked[: min(10, len(ranked))],
    })
entropy_ranking_audit_df = pd.DataFrame(audit_rows)
display(entropy_ranking_audit_df)


## 5. Train All-Imaging Diagnostic Models

These runs compare modelling behaviour on the full imaging pool. The primary workflow should still be read from the newer SRM and comparator notebooks.


In [ ]:
results = []

# LDA visit-separation model using the complete imaging pool only.
res = lda_loocv(
    df_long,
    ALL_FEATURES,
    subject_col=subject_col,
    visit_col="visit",
    selection_fn=None,
    cv_n_splits=CV_N_SPLITS,
    random_seed=RANDOM_SEED,
)
best_single_d = float(single_feature_d_df["d_feature"].max()) if len(single_feature_d_df) else np.nan
results.append({
    "method": "LDA",
    "task": "separation",
    "target": "visit_axis",
    "feature_pool": "all_imaging",
    "n_features": len(ALL_FEATURES),
    "selection_mode": "none",
    "k_selected": np.nan,
    "entropy_source": np.nan,
    "d_score": res["d_score"],
    "srm": res["srm"],
    "d_ci_low": res["d_ci_low"],
    "d_ci_high": res["d_ci_high"],
    "rmse": np.nan,
    "r2": np.nan,
    "n_subjects": res["n_subjects"],
    "beats_best_single": (
        bool(res["d_score"] > best_single_d)
        if np.isfinite(res["d_score"]) and np.isfinite(best_single_d) else np.nan
    ),
    "notes": "all imaging features",
})

results_df = pd.DataFrame(results)
print("Total result rows:", len(results_df))
display(results_df.sort_values("d_score", ascending=False))


## 6. Summarise Diagnostic Results

The final table records available rows from this entropy-focused audit so they can be compared with the dedicated model notebooks.


In [ ]:
print("Results rows:", len(results_df))
sep = results_df[results_df["task"] == "separation"].sort_values("d_score", ascending=False)
print("Top separation rows:", len(sep))
display(sep.head(10))
